# Phase 4 CJPE — Experiment 2/6: Partial fine-tuning / layer freezing

**Technique:** Partial / layer freezing

The embedding and the lower blocks stay frozen; only the top `K` encoder blocks, the final norm and the head train. Lower layers in a masked LM carry general lexical and syntactic structure, upper layers carry task-shaped abstractions — so this asks how much of the network CJPE actually needs to move.

---
### Self-contained
No `.py` files, no setup step. The library cells in section 2 are identical in
all six notebooks, so any of them runs on its own. You only need your own
`a1_tokenizer.py`, `bpe_40k.json` and `best_enc.pt` in Drive.

### How the six fit together
Every notebook shares the same data pipeline, 512-token window, mean-pool +
linear head, training loop and evaluation. The **only** thing that differs is
the adaptation technique — which is what the phase requires ("these experiments
must not consist only of changing hyperparameters of the same method").

Results append to `cjpe_results.json` in Drive, so the comparison table at the
end of any notebook shows every experiment run so far.

**Order:** 01 first (it is the reference the others are compared against, and it
saves the teacher notebook 06 needs), then 02-05 in any order, 06 last.

Set `quick=True` in the config cell for a ~10 minute smoke run before committing
a full GPU hour.

## 1 — Setup

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# ---- EDIT: the folder holding a1_tokenizer.py, bpe_40k.json, best_enc.pt ----
DRIVE = "/content/drive/MyDrive"
# -----------------------------------------------------------------------------

import sys, os, subprocess
sys.path.insert(0, DRIVE)          # so `import a1_tokenizer` finds yours
subprocess.check_call([sys.executable, "-m", "pip", "-q", "install",
                       "huggingface_hub", "scikit-learn"])

for f in ("a1_tokenizer.py", "bpe_40k.json", "best_enc.pt"):
    p = os.path.join(DRIVE, f)
    print(("FOUND    " if os.path.exists(p) else "MISSING  ") + f
          + (f"  {os.path.getsize(p)/1e6:.1f} MB" if os.path.exists(p) else ""))

import torch
print("\ntorch", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU — Runtime > Change runtime type > T4")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
FOUND    a1_tokenizer.py  0.0 MB
FOUND    bpe_40k.json  1.2 MB
FOUND    best_enc.pt  1565.3 MB

torch 2.11.0+cu130
GPU: Tesla T4


## 2 — Library

Everything the experiment needs, defined inline. These six cells are **identical
in all six notebooks**, so whichever one you open is self-contained — there are
no `.py` files to upload and no setup notebook to run first.

Run them once per session (Runtime → Run all does it for you). You can collapse
them; nothing below depends on reading them.

The only file this still needs from your Drive is **`a1_tokenizer.py`**, your own
Phase 1 tokenizer module — it has to be yours so it matches `bpe_40k.json`
exactly.

### Library 1/6 — imports, configuration, tokenizer

In [ ]:
import json, math, os, random, re, sys, time
from dataclasses import dataclass, field, asdict
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

@dataclass
class CJPEConfig:
    # --- paths (override in the notebook) ---
    drive_dir: str = "/content/drive/MyDrive"
    enc_ckpt: str = "/content/drive/MyDrive/best_enc.pt"
    dec_ckpt: str = "/content/drive/MyDrive/best.pt"
    tokenizer_json: str = "/content/drive/MyDrive/bpe_40k.json"
    cache_dir: str = "/content/drive/MyDrive/cjpe_cache"
    results_json: str = "/content/drive/MyDrive/cjpe_results.json"

    # --- document representation (held CONSTANT across the 6 experiments) ---
    max_len: int = 512
    window: str = "head_tail"        # head | tail | head_tail
    head_frac: float = 0.5           # for head_tail: share of max_len from the start

    # --- backbone ---
    n_heads: int = 12
    dropout: float = 0.1
    rope_theta: float = 10000.0

    # --- training ---
    lr: float = 2e-5                 # backbone lr (PEFT notebooks raise this)
    head_lr: float = 1e-3            # classifier head always trains faster
    weight_decay: float = 0.01
    epochs: int = 2
    batch_size: int = 8
    grad_accum: int = 2
    warmup_frac: float = 0.06
    max_grad_norm: float = 1.0
    label_smoothing: float = 0.0
    seed: int = 42
    eval_every: int = 400            # optimizer steps between dev evaluations
    patience: int = 4                # dev evals without macro-F1 improvement

    # --- data size (set quick=True for a fast smoke run) ---
    quick: bool = False
    max_train: int | None = None     # None = use all of train.json
    max_dev: int | None = None

    hf_repo: str = "Exploration-Lab/CS-781-Capstone"
    hf_token: str = ""

    def __post_init__(self):
        if self.quick:
            self.max_train = self.max_train or 3000
            self.max_dev = self.max_dev or 800
            self.epochs = 1
            self.eval_every = 100


def set_seed(s: int):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PAD_NAMES = ["<pad>", "[PAD]", "<PAD>", "[pad]"]


def load_tokenizer(cfg: CJPEConfig, py_dir: str | None = None):
    """Returns (tokenizer, pad_id, base_vocab_size)."""
    d = py_dir or str(Path(cfg.tokenizer_json).parent)
    if d not in sys.path:
        sys.path.insert(0, d)
    from a1_tokenizer import BPETokenizer
    tok = BPETokenizer.load(cfg.tokenizer_json)
    base = int(tok.vocab_size)

    pad = None
    st = getattr(tok, "special_tokens", None)
    if isinstance(st, dict):
        for n in PAD_NAMES:
            if isinstance(st.get(n), int):
                pad = int(st[n]); break
    if pad is None:
        for attr in ("token_to_id", "vocab", "encoder", "stoi"):
            v = getattr(tok, attr, None)
            if isinstance(v, dict):
                for n in PAD_NAMES:
                    if isinstance(v.get(n), int):
                        pad = int(v[n]); break
            if pad is not None:
                break
    if pad is None:
        raise ValueError("Could not resolve the pad id from this tokenizer.")
    return tok, pad, base

### Library 2/6 — data loading, windowing, caching

In [ ]:
def _hf_path(cfg: CJPEConfig, rel: str) -> str:
    from huggingface_hub import hf_hub_download
    return hf_hub_download(cfg.hf_repo, rel, repo_type="dataset", token=cfg.hf_token)


def load_split(cfg: CJPEConfig, split: str, local: str | None = None):
    """split: train | dev | test. Returns list of (id, text, label|None)."""
    if local and os.path.exists(local):
        path = local
    else:
        sub = "Test" if split == "test" else "Train"
        path = _hf_path(cfg, f"Phase_4_CJPE/{sub}/{split}.json")
    with open(path) as f:
        raw = json.load(f)
    rows = raw.values() if isinstance(raw, dict) else raw
    out = []
    for r in rows:
        if not isinstance(r, dict):
            continue
        txt = r.get("text") or r.get("document") or ""
        if isinstance(txt, list):
            txt = " ".join(map(str, txt))
        if not txt:
            continue
        lab = r.get("label", r.get("labels"))
        if isinstance(lab, list):
            lab = lab[0] if lab else None
        out.append((str(r.get("id", len(out))), txt,
                    None if lab is None else int(lab)))
    return out


# ---- windowing: how a long judgment becomes max_len tokens ----

def encode_window(tok, text, pad_id, cfg: CJPEConfig):
    """Returns (ids[max_len], n_real). Pre-truncates by words so the pure-Python
    BPE never processes material that would be discarded: encode() splits on
    whitespace and encodes each word independently, so the ids for the first N
    words are a prefix of the full document's ids (and likewise for a suffix)."""
    L, w = cfg.max_len, text.split()
    budget_words = 6 * L                      # ~4x margin over tokens-per-word

    if cfg.window == "head":
        ids = tok.encode(" ".join(w[:budget_words]), add_special_tokens=True)[:L]
    elif cfg.window == "tail":
        ids = tok.encode(" ".join(w[-budget_words:]), add_special_tokens=True)[-L:]
    elif cfg.window == "head_tail":
        n_head = int(round(L * cfg.head_frac))
        n_tail = L - n_head
        h_full = tok.encode(" ".join(w[:budget_words]), add_special_tokens=True)
        # A document that already fits must NOT be spliced, or the head and the
        # tail slice would both cover it and the model would see it twice.
        if len(w) <= budget_words and len(h_full) <= L:
            ids = h_full
        else:
            t = tok.encode(" ".join(w[-budget_words:]), add_special_tokens=True)
            ids = h_full[:n_head] + t[-n_tail:]
    else:
        raise ValueError(f"unknown window {cfg.window!r}")

    n = max(len(ids), 1)
    if len(ids) < L:
        ids = ids + [pad_id] * (L - len(ids))
    return ids[:L], min(n, L)


def encode_split(tok, rows, pad_id, cfg: CJPEConfig, desc=""):
    X = np.full((len(rows), cfg.max_len), pad_id, dtype=np.int32)
    n = np.zeros(len(rows), dtype=np.int32)
    t0 = time.time()
    for i, (_, text, _) in enumerate(rows):
        ids, k = encode_window(tok, text, pad_id, cfg)
        X[i] = ids; n[i] = k
        if desc and i % 4000 == 0 and i:
            print(f"  {desc} {i}/{len(rows)} ({time.time()-t0:.0f}s)")
    if desc:
        print(f"  {desc} {len(rows)} docs in {time.time()-t0:.0f}s")
    return X, n


def prepare_data(cfg: CJPEConfig, tok, pad_id, splits=("train", "dev"),
                 local_paths: dict | None = None):
    """Tokenizes once and caches to Drive, keyed by window settings, so the
    other five notebooks reuse it instead of re-tokenizing."""
    local_paths = local_paths or {}
    Path(cfg.cache_dir).mkdir(parents=True, exist_ok=True)
    key = f"{cfg.window}_{cfg.max_len}_{cfg.head_frac:.2f}"
    out = {}
    for sp in splits:
        cap = cfg.max_train if sp == "train" else cfg.max_dev
        tag = f"{sp}_{key}" + (f"_n{cap}" if cap else "")
        fx = Path(cfg.cache_dir) / f"X_{tag}.npy"
        fn = Path(cfg.cache_dir) / f"n_{tag}.npy"
        fm = Path(cfg.cache_dir) / f"meta_{tag}.json"
        if fx.exists() and fn.exists() and fm.exists():
            meta = json.loads(fm.read_text())
            out[sp] = (np.load(fx), np.load(fn),
                       np.array(meta["y"]) if meta["y"] is not None else None,
                       meta["ids"])
            print(f"[cache hit ] {sp}: {out[sp][0].shape[0]} docs  ({fx.name})")
            continue
        rows = load_split(cfg, sp, local_paths.get(sp))
        if cap and len(rows) > cap:
            rng = np.random.default_rng(cfg.seed)
            sel = rng.choice(len(rows), cap, replace=False)
            rows = [rows[i] for i in sel]
        print(f"[tokenizing] {sp}: {len(rows)} docs, window={cfg.window}")
        X, n = encode_split(tok, rows, pad_id, cfg, desc=sp)
        y = None if rows[0][2] is None else np.array([r[2] for r in rows], dtype=np.int64)
        ids = [r[0] for r in rows]
        np.save(fx, X); np.save(fn, n)
        fm.write_text(json.dumps({"y": None if y is None else y.tolist(), "ids": ids}))
        out[sp] = (X, n, y, ids)
    return out


class CJPEDataset(torch.utils.data.Dataset):
    def __init__(self, X, n, y=None):
        self.X, self.n, self.y = X, n, y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        item = (torch.from_numpy(self.X[i].astype(np.int64)), int(self.n[i]))
        return item + ((int(self.y[i]),) if self.y is not None else (-1,))


def make_loader(X, n, y, batch_size, shuffle, seed=42):
    g = torch.Generator(); g.manual_seed(seed)
    return torch.utils.data.DataLoader(
        CJPEDataset(X, n, y), batch_size=batch_size, shuffle=shuffle,
        num_workers=2, pin_memory=(DEVICE.type == "cuda"), generator=g,
        drop_last=False)

### Library 3/6 — the Phase 1 encoder and the classifier

In [ ]:
# Copied from encoder_legal_lm_fixed.ipynb so the Phase 1 state_dict loads
# into exactly the code that produced it.

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        rms = torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return (x.float() * rms).to(x.dtype) * self.weight


def precompute_rope(dim, max_len, theta=10000.0, device=None):
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2, device=device).float() / dim))
    t = torch.arange(max_len, device=device).float()
    ang = torch.outer(t, freqs)
    return ang.cos(), ang.sin()


def apply_rope(x, cos, sin):
    T = x.size(2)
    c = cos[:T].unsqueeze(0).unsqueeze(0)
    s = sin[:T].unsqueeze(0).unsqueeze(0)
    hd = x.size(-1) // 2
    x1, x2 = x[..., :hd], x[..., hd:]
    rot = torch.cat([-x2, x1], dim=-1)
    c_e = torch.cat([c.expand(-1, -1, -1, hd), c.expand(-1, -1, -1, hd)], -1)
    s_e = torch.cat([s.expand(-1, -1, -1, hd), s.expand(-1, -1, -1, hd)], -1)
    return x * c_e + rot * s_e


class MultiHeadSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.nh, self.hd, self.dm = cfg.n_heads, cfg.head_dim, cfg.d_model
        self.q = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.k = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.v = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.o = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.attn_drop = nn.Dropout(cfg.dropout)
        self.resid_drop = nn.Dropout(cfg.dropout)
        self.scale = 1.0 / math.sqrt(cfg.head_dim)
    def forward(self, x, cos, sin, padding_mask=None):
        B, T, _ = x.shape
        q = self.q(x).view(B, T, self.nh, self.hd).transpose(1, 2)
        k = self.k(x).view(B, T, self.nh, self.hd).transpose(1, 2)
        v = self.v(x).view(B, T, self.nh, self.hd).transpose(1, 2)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        w = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        if padding_mask is not None:
            w = w.masked_fill(padding_mask.unsqueeze(1).unsqueeze(2), float("-inf"))
        w = F.softmax(w, dim=-1, dtype=torch.float32).to(q.dtype)
        w = self.attn_drop(w)
        out = torch.matmul(w, v).transpose(1, 2).contiguous().view(B, T, self.dm)
        return self.resid_drop(self.o(out))


class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.gate = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.up   = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.down = nn.Linear(cfg.d_ff, cfg.d_model, bias=False)
        self.drop = nn.Dropout(cfg.dropout)
    def forward(self, x):
        return self.drop(self.down(F.silu(self.gate(x)) * self.up(x)))


class EncoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.d_model); self.attn = MultiHeadSelfAttention(cfg)
        self.norm2 = RMSNorm(cfg.d_model); self.ffn  = SwiGLU(cfg)
    def forward(self, x, cos, sin, padding_mask=None):
        x = x + self.attn(self.norm1(x), cos, sin, padding_mask)
        return x + self.ffn(self.norm2(x))


class BackboneCfg:
    pass


class LegalEncoder(nn.Module):
    """Phase 1 encoder without the MLM head."""
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        self.layers = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg.n_layers)])
        self.norm = RMSNorm(cfg.d_model)
        cos, sin = precompute_rope(cfg.head_dim, cfg.max_seq_len, cfg.rope_theta)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        self.apply(self._init)   # only matters for tensors absent from the ckpt

    @staticmethod
    def _init(m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, std=0.02)

    def forward(self, ids, padding_mask=None):
        T = ids.size(1)
        if T > self.rope_cos.size(0):
            raise ValueError(
                f"sequence length {T} exceeds the RoPE table ({self.rope_cos.size(0)}). "
                f"Phase 1 precomputed RoPE at max_seq_len only; longer context needs "
                f"the buffers rebuilt and position interpolation.")
        x = self.drop(self.tok_emb(ids))
        for layer in self.layers:
            x = layer(x, self.rope_cos, self.rope_sin, padding_mask)
        return self.norm(x)


def backbone_cfg_from_sd(sd, n_heads=12, max_seq_len=512, dropout=0.1,
                         rope_theta=10000.0):
    c = BackboneCfg()
    c.vocab_size, c.d_model = sd["tok_emb.weight"].shape
    c.n_layers = max(int(m.group(1)) for k in sd
                     if (m := re.match(r"layers\.(\d+)\.", k))) + 1
    c.d_ff = sd["layers.0.ffn.gate.weight"].shape[0]
    c.n_heads, c.max_seq_len, c.dropout = n_heads, max_seq_len, dropout
    c.rope_theta = rope_theta
    if c.d_model % c.n_heads:
        raise ValueError(f"d_model {c.d_model} not divisible by n_heads {c.n_heads}")
    c.head_dim = c.d_model // c.n_heads
    return c


class CJPEClassifier(nn.Module):
    """Phase 1 encoder + mean-pool + linear head. The head and pooling are the
    SAME in every experiment, so differences come from the adaptation
    technique, not from the readout (which the rules say does not count as a
    technique on its own)."""
    def __init__(self, backbone, d_model, n_classes=2, p_drop=0.1):
        super().__init__()
        self.backbone = backbone
        self.drop = nn.Dropout(p_drop)
        self.head = nn.Linear(d_model, n_classes)
        nn.init.normal_(self.head.weight, std=0.02)
        nn.init.zeros_(self.head.bias)

    def pooled(self, ids, pad_mask):
        h = self.backbone(ids, pad_mask)
        keep = (~pad_mask).unsqueeze(-1).to(h.dtype)
        return (h * keep).sum(1) / keep.sum(1).clamp(min=1)

    def forward(self, ids, pad_mask):
        return self.head(self.drop(self.pooled(ids, pad_mask)))


def load_phase1_encoder(cfg: CJPEConfig, n_layers_keep: int | None = None,
                        verbose=True):
    """Builds the classifier with Phase 1 weights. n_layers_keep < depth takes
    an evenly spaced subset of layers (used for the distillation student)."""
    ck = torch.load(cfg.enc_ckpt, map_location="cpu", weights_only=False)
    sd = {(k[7:] if k.startswith("module.") else k): v
          for k, v in (ck["model"] if "model" in ck else ck).items()}
    bc = backbone_cfg_from_sd(sd, n_heads=cfg.n_heads, max_seq_len=cfg.max_len,
                              dropout=cfg.dropout, rope_theta=cfg.rope_theta)

    keep_idx = list(range(bc.n_layers))
    if n_layers_keep is not None and n_layers_keep < bc.n_layers:
        step = bc.n_layers / n_layers_keep
        keep_idx = [min(bc.n_layers - 1, int(round(i * step)))
                    for i in range(n_layers_keep)]
        bc.n_layers = n_layers_keep

    bb = LegalEncoder(bc)
    want = bb.state_dict()
    new = {}
    for k in want:
        if k.startswith("layers."):
            i = int(k.split(".")[1])
            src = k.replace(f"layers.{i}.", f"layers.{keep_idx[i]}.", 1)
        else:
            src = k
        if src in sd and sd[src].shape == want[k].shape:
            new[k] = sd[src]
    missing = [k for k in want if k not in new]
    bb.load_state_dict(new, strict=False)
    if verbose:
        print(f"backbone: {bc.d_model}d x {bc.n_layers}L x {bc.n_heads}H "
              f"d_ff={bc.d_ff} vocab={bc.vocab_size:,}")
        print(f"  loaded {len(new)}/{len(want)} tensors from "
              f"{Path(cfg.enc_ckpt).name} (step {ck.get('step','?')}, "
              f"val_loss {ck.get('val_loss', float('nan')):.4f})")
        if keep_idx != list(range(len(keep_idx))):
            print(f"  layer subset: {keep_idx}")
        if missing:
            print(f"  NOT loaded (fresh init): {missing}")
        std = bb.tok_emb.weight.std().item()
        print(f"  tok_emb std = {std:.4f} (fresh init ~0.0200)")
        if abs(std - 0.02) < 0.002:
            print("  !! near init — check that the checkpoint really loaded")
    model = CJPEClassifier(bb, bc.d_model, 2, cfg.dropout)
    return model, bc, ck

### Library 4/6 — adaptation modules (LoRA / AdaLoRA / adapters)

In [ ]:
def param_report(model, label=""):
    tot = sum(p.numel() for p in model.parameters())
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"{label}trainable {tr:,} / {tot:,} = {100*tr/max(tot,1):.3f}%")
    return tr, tot


def freeze_all(model):
    for p in model.parameters():
        p.requires_grad = False


def unfreeze_head(model):
    for p in model.head.parameters():
        p.requires_grad = True


class LoRALinear(nn.Module):
    """y = W0 x + (alpha/r) * B(A(x)).  W0 frozen; A, B trainable."""
    def __init__(self, base: nn.Linear, r=8, alpha=16, p_drop=0.0):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False
        self.r, self.alpha = r, alpha
        self.scaling = alpha / r
        self.lora_A = nn.Parameter(torch.zeros(r, base.in_features))
        self.lora_B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)        # ΔW = 0 at init
        self.drop = nn.Dropout(p_drop)

    def forward(self, x):
        out = self.base(x)
        return out + self.drop(x) @ self.lora_A.T @ self.lora_B.T * self.scaling


class AdaLoRALinear(nn.Module):
    """SVD-style ΔW = P diag(Λ) Q with a prunable rank budget.

    Implements the parts of AdaLoRA that carry the idea: the triplet
    parameterisation, a sensitivity-based importance score per rank with
    exponential smoothing, a cubic budget schedule that prunes rank over
    training, and an orthogonality penalty on P and Q. It does NOT implement
    global budget redistribution across modules — each module anneals to the
    same target rank. Say so in the report rather than claiming full AdaLoRA.
    """
    def __init__(self, base: nn.Linear, r_init=12, alpha=16, p_drop=0.0):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False
        self.r_init, self.alpha = r_init, alpha
        self.scaling = alpha / r_init
        self.P = nn.Parameter(torch.zeros(base.out_features, r_init))
        self.E = nn.Parameter(torch.zeros(r_init))
        self.Q = nn.Parameter(torch.zeros(r_init, base.in_features))
        nn.init.normal_(self.P, std=0.02)
        nn.init.normal_(self.Q, std=0.02)
        nn.init.zeros_(self.E)             # ΔW = 0 at init
        self.register_buffer("rank_mask", torch.ones(r_init))
        self.register_buffer("ipt", torch.zeros(r_init))
        self.drop = nn.Dropout(p_drop)

    def forward(self, x):
        e = self.E * self.rank_mask
        return self.base(x) + (self.drop(x) @ self.Q.T * e) @ self.P.T * self.scaling

    @torch.no_grad()
    def update_importance(self, beta=0.85):
        """Sensitivity |λ * dL/dλ|, exponentially smoothed. Call after backward."""
        if self.E.grad is None:
            return
        s = (self.E * self.E.grad).abs()
        self.ipt.mul_(beta).add_(s, alpha=1 - beta)

    @torch.no_grad()
    def set_rank(self, k):
        """Keep the k most important directions."""
        k = max(1, min(int(k), self.r_init))
        keep = torch.topk(self.ipt, k).indices
        m = torch.zeros_like(self.rank_mask)
        m[keep] = 1.0
        self.rank_mask.copy_(m)

    def ortho_penalty(self):
        I_p = torch.eye(self.P.size(1), device=self.P.device)
        I_q = torch.eye(self.Q.size(0), device=self.Q.device)
        return ((self.P.T @ self.P - I_p).pow(2).mean()
                + (self.Q @ self.Q.T - I_q).pow(2).mean())


class Adapter(nn.Module):
    """Houlsby bottleneck: x + Up(GELU(Down(LN(x)))), up zero-init."""
    def __init__(self, d_model, bottleneck=64):
        super().__init__()
        self.norm = RMSNorm(d_model)
        self.down = nn.Linear(d_model, bottleneck)
        self.up = nn.Linear(bottleneck, d_model)
        nn.init.normal_(self.down.weight, std=0.02); nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self, x):
        return x + self.up(F.gelu(self.down(self.norm(x))))


class AdaptedBlock(nn.Module):
    """Wraps a frozen EncoderBlock, inserting an adapter after each sublayer."""
    def __init__(self, block: EncoderBlock, d_model, bottleneck=64):
        super().__init__()
        self.block = block
        self.adapter_attn = Adapter(d_model, bottleneck)
        self.adapter_ffn = Adapter(d_model, bottleneck)
    def forward(self, x, cos, sin, padding_mask=None):
        b = self.block
        x = x + self.adapter_attn(b.attn(b.norm1(x), cos, sin, padding_mask))
        return x + self.adapter_ffn(b.ffn(b.norm2(x)))


# ---- injection helpers ----

ATTN_TARGETS = ("q", "k", "v", "o")
FFN_TARGETS = ("gate", "up", "down")


def inject_lora(model, r=8, alpha=16, targets=ATTN_TARGETS, ffn_targets=(),
                p_drop=0.0, cls=LoRALinear, **kw):
    """Replaces the named nn.Linear modules inside each block. Returns the list
    of injected modules."""
    injected = []
    for blk in model.backbone.layers:
        for name in targets:
            base = getattr(blk.attn, name)
            new = cls(base, r, alpha, p_drop, **kw) if cls is not LoRALinear else \
                  cls(base, r, alpha, p_drop)
            setattr(blk.attn, name, new); injected.append(new)
        for name in ffn_targets:
            base = getattr(blk.ffn, name)
            new = cls(base, r, alpha, p_drop, **kw) if cls is not LoRALinear else \
                  cls(base, r, alpha, p_drop)
            setattr(blk.ffn, name, new); injected.append(new)
    return injected


def inject_adalora(model, r_init=12, alpha=16, targets=ATTN_TARGETS,
                   ffn_targets=(), p_drop=0.0):
    injected = []
    for blk in model.backbone.layers:
        for name in targets:
            new = AdaLoRALinear(getattr(blk.attn, name), r_init, alpha, p_drop)
            setattr(blk.attn, name, new); injected.append(new)
        for name in ffn_targets:
            new = AdaLoRALinear(getattr(blk.ffn, name), r_init, alpha, p_drop)
            setattr(blk.ffn, name, new); injected.append(new)
    return injected


def inject_adapters(model, bottleneck=64):
    d = model.backbone.cfg.d_model
    model.backbone.layers = nn.ModuleList(
        [AdaptedBlock(b, d, bottleneck) for b in model.backbone.layers])
    return [m for m in model.backbone.layers]


def adalora_budget(step, total, r_init, r_target, warmup_frac=0.1,
                   final_frac=0.3):
    """Cubic schedule: hold r_init, prune to r_target, then hold."""
    w, f = warmup_frac * total, (1 - final_frac) * total
    if step <= w:
        return r_init
    if step >= f:
        return r_target
    p = (step - w) / max(f - w, 1)
    return int(round(r_target + (r_init - r_target) * (1 - p) ** 3))

### Library 5/6 — training, evaluation, results, submission

In [ ]:
def macro_metrics(y_true, y_pred):
    from sklearn.metrics import (f1_score, precision_score, recall_score,
                                 accuracy_score, confusion_matrix)
    return {
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "f1_per_class": f1_score(y_true, y_pred, average=None, zero_division=0).tolist(),
        "confusion": confusion_matrix(y_true, y_pred).tolist(),
    }


@torch.no_grad()
def predict(model, loader, pad_id, return_logits=True):
    model.eval()
    P, Y = [], []
    for ids, n, y in loader:
        ids = ids.to(DEVICE, non_blocking=True)
        pm = ids == pad_id
        with torch.amp.autocast(device_type=DEVICE.type,
                                enabled=(DEVICE.type == "cuda")):
            logits = model(ids, pm)
        P.append(logits.float().cpu())
        Y.append(y)
    logits = torch.cat(P).numpy()
    y = torch.cat(Y).numpy()
    return (logits, y) if return_logits else (logits.argmax(1), y)


def evaluate(model, loader, pad_id):
    logits, y = predict(model, loader, pad_id)
    m = macro_metrics(y, logits.argmax(1))
    m["_logits"], m["_y"] = logits, y
    return m


def train_model(model, train_loader, dev_loader, pad_id, cfg: CJPEConfig,
                param_groups=None, class_weight=None, on_backward=None,
                on_step=None, extra_loss=None, label="run"):
    """One training loop for every experiment.

    on_backward(step)      called after loss.backward(), before the optimizer step
    on_step(step, total)   called after each optimizer step (rank schedules etc.)
    extra_loss()           returns a scalar added to the loss (regularisers)
    """
    model.to(DEVICE)
    if param_groups is None:
        decay, no_decay, head = [], [], []
        for n_, p in model.named_parameters():
            if not p.requires_grad:
                continue
            if n_.startswith("head."):
                head.append(p)
            elif p.ndim < 2 or "norm" in n_.lower() or n_.endswith(".bias"):
                no_decay.append(p)
            else:
                decay.append(p)
        param_groups = [
            {"params": decay, "weight_decay": cfg.weight_decay, "lr": cfg.lr},
            {"params": no_decay, "weight_decay": 0.0, "lr": cfg.lr},
            {"params": head, "weight_decay": cfg.weight_decay, "lr": cfg.head_lr},
        ]
    param_groups = [g for g in param_groups if len(g["params"])]
    opt = torch.optim.AdamW(param_groups, betas=(0.9, 0.999))
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))

    steps_per_epoch = max(1, math.ceil(len(train_loader) / cfg.grad_accum))
    total = steps_per_epoch * cfg.epochs
    warm = max(1, int(cfg.warmup_frac * total))
    base_lrs = [g["lr"] for g in opt.param_groups]

    def lr_at(s):
        if s < warm:
            return (s + 1) / warm
        p = (s - warm) / max(1, total - warm)
        return 0.5 * (1 + math.cos(math.pi * min(p, 1.0)))

    cw = None if class_weight is None else torch.tensor(
        class_weight, dtype=torch.float32, device=DEVICE)

    hist, best, best_state, no_imp, step = [], -1.0, None, 0, 0
    t0 = time.time()
    micro = 0
    model.train()
    stop = False
    for ep in range(cfg.epochs):
        if stop:
            break
        for ids, n, y in train_loader:
            ids = ids.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            pm = ids == pad_id
            with torch.amp.autocast(device_type=DEVICE.type,
                                    enabled=(DEVICE.type == "cuda")):
                logits = model(ids, pm)
                loss = F.cross_entropy(logits.float(), y, weight=cw,
                                       label_smoothing=cfg.label_smoothing)
                if extra_loss is not None:
                    loss = loss + extra_loss()
                loss = loss / cfg.grad_accum
            scaler.scale(loss).backward()
            micro += 1
            if micro % cfg.grad_accum:
                continue

            scaler.unscale_(opt)
            if on_backward is not None:
                on_backward(step)
            torch.nn.utils.clip_grad_norm_(
                [p for g in opt.param_groups for p in g["params"]],
                cfg.max_grad_norm)
            f = lr_at(step)
            for g, b in zip(opt.param_groups, base_lrs):
                g["lr"] = b * f
            scaler.step(opt); scaler.update()
            opt.zero_grad(set_to_none=True)
            step += 1
            if on_step is not None:
                on_step(step, total)

            if step % cfg.eval_every == 0 or step == total:
                m = evaluate(model, dev_loader, pad_id)
                hist.append({"step": step, "macro_f1": m["macro_f1"],
                             "accuracy": m["accuracy"]})
                flag = ""
                if m["macro_f1"] > best:
                    best = m["macro_f1"]; no_imp = 0
                    best_state = {k: v.detach().cpu().clone()
                                  for k, v in model.state_dict().items()}
                    flag = "  *best*"
                else:
                    no_imp += 1
                print(f"  [{label}] step {step}/{total} ep{ep+1} "
                      f"dev macro-F1 {m['macro_f1']:.4f} acc {m['accuracy']:.4f}"
                      f"{flag}  ({time.time()-t0:.0f}s)")
                model.train()
                if no_imp >= cfg.patience:
                    print(f"  [{label}] early stop: {cfg.patience} evals without "
                          f"improvement")
                    stop = True
                    break

    if best_state is not None:
        model.load_state_dict(best_state)
    return {"best_dev_macro_f1": best, "history": hist,
            "train_seconds": time.time() - t0, "steps": step}

def record_result(cfg: CJPEConfig, name, technique, metrics, trainable, total,
                  train_info, notes="", extra=None):
    path = Path(cfg.results_json)
    allr = json.loads(path.read_text()) if path.exists() else []
    allr = [r for r in allr if r.get("name") != name]
    row = {
        "name": name, "technique": technique,
        "macro_f1": metrics["macro_f1"], "precision": metrics["precision"],
        "recall": metrics["recall"], "accuracy": metrics["accuracy"],
        "f1_per_class": metrics["f1_per_class"], "confusion": metrics["confusion"],
        "trainable_params": int(trainable), "total_params": int(total),
        "trainable_pct": round(100 * trainable / max(total, 1), 4),
        "train_seconds": round(train_info.get("train_seconds", 0), 1),
        "steps": train_info.get("steps", 0),
        "history": train_info.get("history", []),
        "window": cfg.window, "max_len": cfg.max_len, "epochs": cfg.epochs,
        "lr": cfg.lr, "batch": cfg.batch_size * cfg.grad_accum,
        "quick": cfg.quick, "n_train": cfg.max_train, "notes": notes,
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    if extra:
        row.update(extra)
    allr.append(row)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(allr, indent=1))
    print(f"recorded '{name}' -> {path}")
    return row


def save_dev_predictions(cfg: CJPEConfig, name, ids, logits, y):
    p = Path(cfg.cache_dir) / f"devpred_{name}.npz"
    np.savez(p, ids=np.array(ids), logits=logits, y=y)
    return str(p)


def load_results(cfg: CJPEConfig):
    path = Path(cfg.results_json)
    return json.loads(path.read_text()) if path.exists() else []

def write_submission(cfg: CJPEConfig, model, tok, pad_id, out_path,
                     local_test=None, batch_size=16):
    """Writes [{"id": ..., "pred": int}, ...] for every case in test.json."""
    rows = load_split(cfg, "test", local_test)
    print(f"test cases: {len(rows)}")
    X, n = encode_split(tok, rows, pad_id, cfg, desc="test")
    loader = make_loader(X, n, None, batch_size, shuffle=False)
    logits, _ = predict(model, loader, pad_id)
    pred = logits.argmax(1).astype(int)
    sub = [{"id": rows[i][0], "pred": int(pred[i])} for i in range(len(rows))]
    assert len(sub) == len(rows), "one prediction per test case is required"
    assert all(s["pred"] in (0, 1) for s in sub), "pred must be 0 or 1"
    Path(out_path).write_text(json.dumps(sub, indent=1))
    from collections import Counter
    print(f"wrote {out_path}  ({len(sub)} rows, pred mix {Counter(pred.tolist())})")
    return sub

### Library 6/6 — comparison and analytics

In [ ]:
import json, math
from pathlib import Path

import numpy as np



# ════════════════════════════════════════════════════ leaderboard

def leaderboard(cfg: CJPEConfig, sort="macro_f1"):
    rows = load_results(cfg)
    if not rows:
        print("No results recorded yet.")
        return []
    rows = sorted(rows, key=lambda r: -r.get(sort, 0))
    w = max(len(r["name"]) for r in rows) + 2
    print("=" * (w + 62))
    print("PHASE 4 CJPE — EXPERIMENT COMPARISON (dev set)")
    print("=" * (w + 62))
    print(f"{'experiment':<{w}}{'macro-F1':>9}{'prec':>7}{'rec':>7}"
          f"{'acc':>7}{'train %':>9}{'params':>12}{'mins':>6}")
    print("-" * (w + 62))
    for r in rows:
        print(f"{r['name']:<{w}}{r['macro_f1']:>9.4f}{r['precision']:>7.4f}"
              f"{r['recall']:>7.4f}{r['accuracy']:>7.4f}"
              f"{r['trainable_pct']:>9.3f}{r['trainable_params']:>12,}"
              f"{r['train_seconds']/60:>6.1f}")
    print("-" * (w + 62))
    n = rows[0].get("n_dev") or 1988
    print(f"95% CI half-width on accuracy at n={n}: "
          f"+/-{1.96*math.sqrt(0.25/n):.4f} — gaps smaller than this are noise")
    return rows


def vs_baseline(cfg: CJPEConfig, baseline="01_full_finetune"):
    """Delta of every run against the full fine-tuning reference. This is the
    'did the technique help' question the phase asks about PEFT."""
    rows = {r["name"]: r for r in load_results(cfg)}
    if baseline not in rows:
        print(f"baseline '{baseline}' not recorded yet — run that notebook first.")
        return
    b = rows[baseline]
    print(f"\nRelative to {baseline} (macro-F1 {b['macro_f1']:.4f}, "
          f"{b['trainable_params']:,} trainable):")
    print(f"{'experiment':<26}{'d macro-F1':>12}{'params vs base':>16}{'retained':>10}")
    print("-" * 64)
    for name, r in sorted(rows.items(), key=lambda kv: -kv[1]["macro_f1"]):
        if name == baseline:
            continue
        d = r["macro_f1"] - b["macro_f1"]
        ratio = r["trainable_params"] / max(b["trainable_params"], 1)
        ret = 100 * r["macro_f1"] / max(b["macro_f1"], 1e-9)
        print(f"{name:<26}{d:>+12.4f}{ratio:>15.3f}x{ret:>9.1f}%")
    print("\n'retained' = this run's macro-F1 as a percentage of full "
          "fine-tuning's.\nA PEFT method near 100% with a tiny parameter ratio "
          "is the result\nthe phase is asking you to look for.")


# ════════════════════════════════════════════════════ significance

def mcnemar(cfg: CJPEConfig, name_a, name_b):
    """Paired test on the same dev cases — far more sensitive than comparing
    two accuracies, because it only looks at cases where the models disagree."""
    pa, pb = _load_pred(cfg, name_a), _load_pred(cfg, name_b)
    if pa is None or pb is None:
        print("missing saved dev predictions for one of these runs")
        return
    ya, yb = pa["y"], pb["y"]
    assert np.array_equal(ya, yb), "runs were evaluated on different dev rows"
    a_ok = pa["logits"].argmax(1) == ya
    b_ok = pb["logits"].argmax(1) == yb
    n01 = int((~a_ok & b_ok).sum())      # only B right
    n10 = int((a_ok & ~b_ok).sum())      # only A right
    n = n01 + n10
    if n == 0:
        print(f"{name_a} vs {name_b}: identical predictions")
        return
    chi2 = (abs(n10 - n01) - 1) ** 2 / n          # continuity-corrected
    p = math.erfc(math.sqrt(chi2 / 2))
    print(f"\nMcNemar  {name_a}  vs  {name_b}")
    print(f"  only {name_a} correct : {n10}")
    print(f"  only {name_b} correct : {n01}")
    print(f"  chi2 = {chi2:.3f}   p = {p:.4f}   "
          f"{'SIGNIFICANT at 0.05' if p < 0.05 else 'not significant'}")
    return p


def _load_pred(cfg: CJPEConfig, name):
    p = Path(cfg.cache_dir) / f"devpred_{name}.npz"
    if not p.exists():
        return None
    d = np.load(p, allow_pickle=True)
    return {"ids": d["ids"], "logits": d["logits"], "y": d["y"]}


# ════════════════════════════════════════════════════ error analysis

def error_analysis(cfg: CJPEConfig, name, dev_rows=None, n_show=5):
    """Confusion, per-class F1, calibration, and (with dev_rows) how accuracy
    varies with document length — which is the long-document question."""
    d = _load_pred(cfg, name)
    if d is None:
        print(f"no saved dev predictions for '{name}'")
        return
    logits, y = d["logits"], d["y"]
    pred = logits.argmax(1)
    prob = _softmax(logits)[:, 1]

    print(f"\n{'='*58}\nERROR ANALYSIS — {name}\n{'='*58}")
    cm = np.zeros((2, 2), dtype=int)
    for t, p in zip(y, pred):
        cm[t, p] += 1
    print("confusion (rows = true, cols = predicted)")
    print(f"            pred 0   pred 1")
    print(f"  true 0  {cm[0,0]:>8} {cm[0,1]:>8}")
    print(f"  true 1  {cm[1,0]:>8} {cm[1,1]:>8}")
    fp, fn = cm[0, 1], cm[1, 0]
    print(f"\nfalse positives (predicted 1, truly 0): {fp}")
    print(f"false negatives (predicted 0, truly 1): {fn}")
    if fp + fn:
        print(f"error skew: {100*fp/max(fp+fn,1):.1f}% of errors are false positives")

    for c in (0, 1):
        tp = cm[c, c]; fpc = cm[1-c, c]; fnc = cm[c, 1-c]
        prec = tp / max(tp + fpc, 1); rec = tp / max(tp + fnc, 1)
        f1 = 2 * prec * rec / max(prec + rec, 1e-9)
        print(f"  class {c}: P {prec:.4f}  R {rec:.4f}  F1 {f1:.4f}  "
              f"(support {cm[c].sum()})")

    # calibration
    print("\ncalibration (predicted P(class=1) vs observed rate)")
    edges = np.linspace(0, 1, 11)
    ece = 0.0
    for i in range(10):
        m = (prob >= edges[i]) & (prob < edges[i + 1] + (i == 9) * 1e-9)
        if m.sum() == 0:
            continue
        conf = prob[m].mean(); acc = (y[m] == 1).mean()
        ece += m.sum() / len(y) * abs(conf - acc)
        bar = "#" * int(40 * acc)
        print(f"  [{edges[i]:.1f},{edges[i+1]:.1f})  n={m.sum():>5}  "
              f"obs {acc:.3f}  {bar}")
    print(f"  expected calibration error = {ece:.4f}")

    # confidence of errors
    conf_all = np.maximum(prob, 1 - prob)
    wrong = pred != y
    if wrong.any():
        print(f"\nmean confidence: correct {conf_all[~wrong].mean():.3f}  "
              f"wrong {conf_all[wrong].mean():.3f}")
        print("  (wrong predictions made with high confidence are the "
              "dangerous ones)")

    # length buckets
    if dev_rows is not None:
        idmap = {r[0]: r[1] for r in dev_rows}
        wl = np.array([len(idmap.get(str(i), "").split()) for i in d["ids"]])
        if (wl > 0).any():
            print("\naccuracy by document length (the long-document question)")
            qs = sorted(set(np.quantile(wl[wl > 0], [0, .25, .5, .75, 1.0]).tolist()))
            if len(qs) < 2:
                print(f"  all dev documents are {int(qs[0])} words — no spread "
                      f"to bucket")
            else:
                for i in range(len(qs) - 1):
                    lo, hi = qs[i], qs[i + 1]
                    last = i == len(qs) - 2
                    m = (wl >= lo) & ((wl <= hi) if last else (wl < hi)) & (wl > 0)
                    if m.sum() == 0:
                        continue
                    print(f"  {int(lo):>6}-{int(hi):>6} words  n={m.sum():>5}  "
                          f"acc {(pred[m]==y[m]).mean():.4f}")
                print("  a drop in the longest bucket means the 512-token window")
                print("  is missing decisive material in those cases")
    return {"confusion": cm.tolist(), "ece": float(ece)}


def _softmax(z):
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)


# ════════════════════════════════════════════════════ ensembling check

def agreement_matrix(cfg: CJPEConfig, names=None):
    """How much do the techniques actually differ? Low pairwise agreement with
    similar scores means they are making different mistakes — which is what
    makes an ensemble worth trying."""
    rows = load_results(cfg)
    names = names or [r["name"] for r in rows]
    preds = {}
    for n in names:
        d = _load_pred(cfg, n)
        if d is not None:
            preds[n] = (d["logits"].argmax(1), d["y"])
    if len(preds) < 2:
        print("need at least two runs with saved dev predictions")
        return
    ks = list(preds)
    w = max(len(k) for k in ks) + 2
    print(f"\npairwise prediction agreement\n{'':<{w}}" +
          "".join(f"{k[:8]:>10}" for k in ks))
    for a in ks:
        line = f"{a:<{w}}"
        for b in ks:
            line += f"{(preds[a][0] == preds[b][0]).mean():>10.3f}"
        print(line)

    # majority-vote ensemble
    P = np.stack([preds[k][0] for k in ks])
    y = preds[ks[0]][1]
    vote = (P.mean(0) >= 0.5).astype(int)
    from sklearn.metrics import f1_score
    best = max(f1_score(y, preds[k][0], average="macro", zero_division=0) for k in ks)
    ens = f1_score(y, vote, average="macro", zero_division=0)
    print(f"\nmajority vote of {len(ks)} runs: macro-F1 {ens:.4f}  "
          f"(best single {best:.4f}, delta {ens-best:+.4f})")
    print("An ensemble is not one of the six required techniques, but if it")
    print("clearly beats every member that is worth reporting.")


# ════════════════════════════════════════════════════ one-call report

def report_all(cfg: CJPEConfig, dev_rows=None, baseline="01_full_finetune",
               focus=None):
    rows = leaderboard(cfg)
    if not rows:
        return
    vs_baseline(cfg, baseline)
    if focus:
        error_analysis(cfg, focus, dev_rows)
        if baseline in {r["name"] for r in rows} and focus != baseline:
            mcnemar(cfg, baseline, focus)
    if len(rows) >= 2:
        agreement_matrix(cfg)
    print("\nWhat to take from this table is yours to write: which techniques")
    print("worked, whether the parameter-efficient ones kept full fine-tuning's")
    print("performance, and what the error profile says about the 512-token")
    print("window are exactly the questions Phase 4's analysis section asks.")

## 3 — Configuration

In [ ]:
cfg = CJPEConfig(
    drive_dir      = DRIVE,
    enc_ckpt       = f"{DRIVE}/best_enc.pt",
    tokenizer_json = f"{DRIVE}/bpe_40k.json",
    cache_dir      = f"{DRIVE}/cjpe_cache",
    results_json   = f"{DRIVE}/cjpe_results.json",

    # ---- held CONSTANT across all six experiments ----
    window     = "head_tail",   # see the ablation in notebook 01
    max_len    = 512,
    head_frac  = 0.5,

    # ---- set quick=True for a ~10 min smoke run first ----
    quick      = False,
    max_train  = None,          # None = all 37,387 training cases
    epochs     = 2,
    batch_size = 8,
    grad_accum = 2,
)

cfg.lr      = 5e-5      # fewer weights move, so a larger step is safe
cfg.head_lr = 1e-3
UNFREEZE_TOP_K = 4      # how many top blocks to train

set_seed(cfg.seed)
tokenizer, PAD_ID, BASE_VOCAB = load_tokenizer(cfg)
print(f"tokenizer vocab {BASE_VOCAB:,} | PAD_ID {PAD_ID} | "
      f"decode -> {tokenizer.decode([PAD_ID])!r} (empty is correct)")

tokenizer vocab 40,000 | PAD_ID 0 | decode -> '' (empty is correct)


## 4 — Data

Tokenised once and cached to Drive; every other notebook reuses the same arrays.

In [ ]:
data = prepare_data(cfg, tokenizer, PAD_ID, ("train", "dev"))
Xtr, ntr, ytr, idtr = data["train"]
Xde, nde, yde, idde = data["dev"]

print(f"train {Xtr.shape[0]:,} docs (pos rate {ytr.mean():.1%})")
print(f"dev   {Xde.shape[0]:,} docs (pos rate {yde.mean():.1%})")

train_loader = make_loader(Xtr, ntr, ytr, cfg.batch_size, True, cfg.seed)
dev_loader   = make_loader(Xde, nde, yde, cfg.batch_size * 2, False)

counts = np.bincount(ytr, minlength=2)
CLASS_W = (counts.sum() / (2 * np.maximum(counts, 1))).tolist()
print(f"train class counts {counts.tolist()} -> CE weights "
      f"[{CLASS_W[0]:.3f}, {CLASS_W[1]:.3f}]")

Phase_4_CJPE/Train/train.json: reconstructing file:   0%|          |  0.00B /  737MB            

Phase_4_CJPE/Train/train.json: downloading bytes:           |  0.00B            

[tokenizing] train: 37387 docs, window=head_tail
  train 4000/37387 (22s)
  train 8000/37387 (37s)
  train 12000/37387 (52s)
  train 16000/37387 (67s)
  train 20000/37387 (86s)
  train 24000/37387 (104s)
  train 28000/37387 (118s)
  train 32000/37387 (130s)
  train 36000/37387 (147s)
  train 37387 docs in 153s


Phase_4_CJPE/Train/dev.json: reconstructing file:   0%|          |  0.00B / 45.0MB            

Phase_4_CJPE/Train/dev.json: downloading bytes:           |  0.00B            

[tokenizing] dev: 1988 docs, window=head_tail
  dev 1988 docs in 10s
train 37,387 docs (pos rate 41.0%)
dev   1,988 docs (pos rate 50.0%)
train class counts [22067, 15320] -> CE weights [0.847, 1.220]


## 5 — Partial fine-tuning / layer freezing

Freezing the embedding matters more than it looks: tied to a 40,001-token vocabulary it is ~30.7M of the ~129.8M parameters, so freezing it alone removes nearly a quarter of the optimiser state and the memory that goes with it.

In [ ]:
model, bcfg, ck = load_phase1_encoder(cfg)
freeze_all(model)
unfreeze_head(model)

n_layers = len(model.backbone.layers)
K = min(UNFREEZE_TOP_K, n_layers)
for blk in model.backbone.layers[n_layers - K:]:
    for p in blk.parameters():
        p.requires_grad = True
for p in model.backbone.norm.parameters():
    p.requires_grad = True

print(f"training the top {K} of {n_layers} blocks + final norm + head")
print(f"frozen: embedding ({model.backbone.tok_emb.weight.numel():,} params) "
      f"and blocks 0-{n_layers-K-1}")
model.to(DEVICE)
param_report(model, "layer freezing: ")

set_seed(cfg.seed)
info = train_model(model, train_loader, dev_loader, PAD_ID, cfg,
                   class_weight=CLASS_W, label="02_freeze")

backbone: 768d x 14L x 12H d_ff=2048 vocab=40,001
  loaded 128/128 tensors from best_enc.pt (step 29000, val_loss 1.6970)
  tok_emb std = 0.0267 (fresh init ~0.0200)
training the top 4 of 14 blocks + final norm + head
frozen: embedding (30,720,768 params) and blocks 0-9
layer freezing: trainable 28,320,002 / 129,835,010 = 21.812%
  [02_freeze] step 400/4674 ep1 dev macro-F1 0.4735 acc 0.5644  *best*  (220s)
  [02_freeze] step 800/4674 ep1 dev macro-F1 0.5505 acc 0.6046  *best*  (441s)
  [02_freeze] step 1200/4674 ep1 dev macro-F1 0.4900 acc 0.5543  (661s)
  [02_freeze] step 1600/4674 ep1 dev macro-F1 0.5394 acc 0.5895  (882s)
  [02_freeze] step 2000/4674 ep1 dev macro-F1 0.6160 acc 0.6288  *best*  (1104s)
  [02_freeze] step 2400/4674 ep2 dev macro-F1 0.6324 acc 0.6509  *best*  (1325s)
  [02_freeze] step 2800/4674 ep2 dev macro-F1 0.6603 acc 0.6610  *best*  (1547s)
  [02_freeze] step 3200/4674 ep2 dev macro-F1 0.6470 acc 0.6569  (1767s)
  [02_freeze] step 3600/4674 ep2 dev macro-F1 0.66

## 6 — Evaluate and record

In [ ]:
metrics = evaluate(model, dev_loader, PAD_ID)
trainable, total = param_report(model, "final: ")

print(f"\ndev macro-F1 {metrics['macro_f1']:.4f} | "
      f"precision {metrics['precision']:.4f} | recall {metrics['recall']:.4f} | "
      f"accuracy {metrics['accuracy']:.4f}")
print("per-class F1:", [round(x, 4) for x in metrics["f1_per_class"]])
print("confusion   :", metrics["confusion"])

record_result(cfg, "02_layer_freeze", "Partial / layer freezing", metrics,
              trainable, total, info, notes="top-K blocks + norm + head")
save_dev_predictions(cfg, "02_layer_freeze", idde, metrics["_logits"], metrics["_y"])

final: trainable 28,320,002 / 129,835,010 = 21.812%

dev macro-F1 0.6830 | precision 0.6833 | recall 0.6831 | accuracy 0.6831
per-class F1: [0.6884, 0.6776]
confusion   : [[696, 298], [332, 662]]
recorded '02_layer_freeze' -> /content/drive/MyDrive/cjpe_results.json


'/content/drive/MyDrive/cjpe_cache/devpred_02_layer_freeze.npz'

## 7 — Test submission

Only for runs you actually submit; each write costs a few minutes tokenising `test.json`.

In [ ]:
WRITE_SUBMISSION = True      # set False to skip (saves a few minutes)

if WRITE_SUBMISSION:
    out = f"{DRIVE}/submission_02_layer_freeze.json"
    sub = write_submission(cfg, model, tokenizer, PAD_ID, out)
    print(sub[:3])
else:
    print("skipped")

Phase_4_CJPE/Test/test.json: reconstructing file:   0%|          |  0.00B / 35.3MB            

Phase_4_CJPE/Test/test.json: downloading bytes:           |  0.00B            

test cases: 1517
  test 1517 docs in 7s
wrote /content/drive/MyDrive/submission_02_layer_freeze.json  (1517 rows, pred mix Counter({1: 775, 0: 742}))
[{'id': '1986_397', 'pred': 1}, {'id': '1977_183', 'pred': 0}, {'id': '1993_98', 'pred': 1}]


## 8 — Comparison and analytics

Reads every experiment recorded so far.

In [ ]:
dev_rows = load_split(cfg, "dev")
report_all(cfg, dev_rows=dev_rows, baseline="01_full_finetune",
           focus="02_layer_freeze")

PHASE 4 CJPE — EXPERIMENT COMPARISON (dev set)
experiment        macro-F1   prec    rec    acc  train %      params  mins
-------------------------------------------------------------------------------
02_layer_freeze     0.6830 0.6833 0.6831 0.6831   21.812  28,320,002  43.2
-------------------------------------------------------------------------------
95% CI half-width on accuracy at n=1988: +/-0.0220 — gaps smaller than this are noise
baseline '01_full_finetune' not recorded yet — run that notebook first.

ERROR ANALYSIS — 02_layer_freeze
confusion (rows = true, cols = predicted)
            pred 0   pred 1
  true 0       696      298
  true 1       332      662

false positives (predicted 1, truly 0): 298
false negatives (predicted 0, truly 1): 332
error skew: 47.3% of errors are false positives
  class 0: P 0.6770  R 0.7002  F1 0.6884  (support 994)
  class 1: P 0.6896  R 0.6660  F1 0.6776  (support 994)

calibration (predicted P(class=1) vs observed rate)
  [0.0,0.1)  n=   88  o

In [ ]:
# ── paste into your running 02 notebook, after training finished ──
import os, json, torch

OUT_DIR = f"{DRIVE}/cs781_phase4"
os.makedirs(OUT_DIR, exist_ok=True)

NAME = "02_layer_freeze"
path = f"{OUT_DIR}/{NAME}.pt"

torch.save({
    "model":      model.state_dict(),          # best checkpoint (train_model reloads it)
    "name":       NAME,
    "technique":  "Partial / layer freezing",
    "backbone":   vars(bcfg),                  # d_model / n_layers / n_heads / d_ff / vocab
    "window":     cfg.window,
    "max_len":    cfg.max_len,
    "head_frac":  cfg.head_frac,
    "dev_macro_f1": info.get("best_dev_macro_f1"),
    "history":    info.get("history"),
    "trainable":  sum(p.numel() for p in model.parameters() if p.requires_grad),
    "total":      sum(p.numel() for p in model.parameters()),
}, path)

print(f"saved {path}  ({os.path.getsize(path)/1e6:.1f} MB)")
print(f"dev macro-F1 {info.get('best_dev_macro_f1'):.4f}")
print("\nreload later with:")
print(f'  ck = torch.load("{path}", map_location="cpu")')
print( '  model.load_state_dict(ck["model"])')

saved /content/drive/MyDrive/cs781_phase4/02_layer_freeze.pt  (519.4 MB)
dev macro-F1 0.6830

reload later with:
  ck = torch.load("/content/drive/MyDrive/cs781_phase4/02_layer_freeze.pt", map_location="cpu")
  model.load_state_dict(ck["model"])


In [ ]:
"""
Analytics for one CJPE experiment.

The question this answers is "did THIS experiment do anything?" — not how it
ranks against the others. It compares the run against the references that make
that question meaningful (the trivial floors, its own starting point, its own
training curve) and then looks at where it fails.

A cross-experiment leaderboard is available at the bottom as an opt-in, for
when all six are done and you are writing the comparison section.
"""
import json, math
from pathlib import Path

import numpy as np


# ════════════════════════════════════════════════════ helpers

def _softmax(z):
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)


def _load_pred(cfg, name):
    p = Path(cfg.cache_dir) / f"devpred_{name}.npz"
    if not p.exists():
        return None
    d = np.load(p, allow_pickle=True)
    return {"ids": d["ids"], "logits": d["logits"], "y": d["y"]}


def _macro_f1(y, p):
    from sklearn.metrics import f1_score
    return float(f1_score(y, p, average="macro", zero_division=0))


# ════════════════════════════════════════════════════ 1. did it learn?

def training_curve(info, metrics=None):
    """Dev macro-F1 against optimizer step, as a text plot."""
    h = info.get("history") or []
    if not h:
        print("no eval history recorded")
        return
    print("DID IT LEARN?  dev macro-F1 over training")
    print("-" * 58)
    vals = [e["macro_f1"] for e in h]
    lo, hi = min(vals), max(vals)
    span = max(hi - lo, 1e-6)
    for e in h:
        bar = "#" * int(1 + 34 * (e["macro_f1"] - lo) / span)
        star = "  <- best" if e["macro_f1"] == hi else ""
        print(f"  step {e['step']:>6}  {e['macro_f1']:.4f}  {bar}{star}")
    first, best = vals[0], hi
    print("-" * 58)
    print(f"  first eval {first:.4f}  ->  best {best:.4f}   "
          f"change {best - first:+.4f}")
    if best - first < 0.01:
        print("  the run barely moved from its first evaluation")
    last_third = vals[max(1, 2 * len(vals) // 3):]
    if last_third and max(last_third) < hi - 1e-9:
        print("  best was NOT in the final third — it was still improving "
              "earlier, then fell back")
    return {"first": first, "best": best}


# ════════════════════════════════════════════════════ 2. vs the floors

def vs_floors(metrics, y):
    """Is this better than predicting nothing? The references that do not
    depend on any other experiment."""
    n = len(y)
    maj = np.full(n, int(round(y.mean())))
    maj_f1 = _macro_f1(y, maj)
    rng = np.random.default_rng(0)
    coin = np.array([_macro_f1(y, rng.integers(0, 2, n)) for _ in range(40)])

    print("\nIS IT BETTER THAN NOTHING?")
    print("-" * 58)
    print(f"  this experiment        macro-F1 {metrics['macro_f1']:.4f}")
    print(f"  always-majority-class  macro-F1 {maj_f1:.4f}   "
          f"({metrics['macro_f1'] - maj_f1:+.4f})")
    print(f"  random coin flip       macro-F1 {coin.mean():.4f} "
          f"+/- {coin.std():.4f}  ({metrics['macro_f1'] - coin.mean():+.4f})")
    half = 1.96 * math.sqrt(0.25 / n)
    print(f"\n  95% CI half-width on accuracy at n={n}: +/-{half:.4f}")
    print(f"  so a macro-F1 gap under ~{half:.3f} is not distinguishable "
          f"from noise")
    return {"majority": maj_f1, "coin": float(coin.mean())}


# ════════════════════════════════════════════════════ 3. is it balanced?

def balance_report(metrics, y):
    """On a balanced dev set macro-F1 == accuracy for a balanced predictor.
    A gap means the model is favouring one class, which is usually the single
    biggest recoverable loss of macro-F1."""
    cm = np.array(metrics["confusion"])
    pred_rate = cm[:, 1].sum() / cm.sum()
    true_rate = cm[1].sum() / cm.sum()
    gap = metrics["accuracy"] - metrics["macro_f1"]

    print("\nIS IT BALANCED?")
    print("-" * 58)
    print(f"  true  positives in dev : {true_rate:.1%}")
    print(f"  predicted as positive  : {pred_rate:.1%}")
    print(f"  accuracy {metrics['accuracy']:.4f} - macro-F1 "
          f"{metrics['macro_f1']:.4f} = {gap:+.4f}")
    if gap > 0.03:
        print("  a gap this size means predictions are skewed toward one class;")
        print("  the threshold sweep below says how much that is costing you")
    else:
        print("  predictions are close to balanced")
    for c in (0, 1):
        tp = cm[c, c]; fp = cm[1 - c, c]; fn = cm[c, 1 - c]
        pr = tp / max(tp + fp, 1); rc = tp / max(tp + fn, 1)
        print(f"  class {c}: precision {pr:.4f}  recall {rc:.4f}  "
              f"support {cm[c].sum()}")


# ════════════════════════════════════════════════════ 4. threshold sweep

def threshold_sweep(logits, y, show=9):
    """Phase 4 explicitly allows calibration and post-processing. This asks
    what macro-F1 the SAME model would reach at a different decision
    threshold — a free gain that costs no retraining."""
    prob = _softmax(logits)[:, 1]
    ts = np.linspace(0.05, 0.95, 91)
    f1s = np.array([_macro_f1(y, (prob >= t).astype(int)) for t in ts])
    i = int(f1s.argmax())
    at_half = _macro_f1(y, (prob >= 0.5).astype(int))

    print("\nWOULD A DIFFERENT THRESHOLD HELP?")
    print("-" * 58)
    step = max(1, len(ts) // show)
    for j in range(0, len(ts), step):
        mark = "  <- best" if j == i else ("  <- argmax (default)"
                                           if abs(ts[j] - 0.5) < 1e-9 else "")
        bar = "#" * int(40 * f1s[j])
        print(f"  t={ts[j]:.2f}  macro-F1 {f1s[j]:.4f}  {bar}{mark}")
    print("-" * 58)
    print(f"  default t=0.50 : {at_half:.4f}")
    print(f"  best    t={ts[i]:.2f} : {f1s[i]:.4f}   ({f1s[i] - at_half:+.4f})")
    if f1s[i] - at_half > 0.01:
        print("  worth reporting: tuning the threshold on dev is legitimate")
        print("  post-processing under Phase 4's rules, and costs nothing.")
        print("  Fit it on dev, then apply the SAME value to test.")
    return {"best_t": float(ts[i]), "f1_at_best_t": float(f1s[i]),
            "f1_at_half": float(at_half)}


# ════════════════════════════════════════════════════ 5. where does it fail?

def error_profile(logits, y, ids=None, dev_rows=None):
    pred = logits.argmax(1)
    prob = _softmax(logits)[:, 1]
    cm = np.zeros((2, 2), dtype=int)
    for t, p in zip(y, pred):
        cm[t, p] += 1

    print("\nWHERE DOES IT FAIL?")
    print("-" * 58)
    print("  confusion (rows = true, cols = predicted)")
    print(f"              pred 0   pred 1")
    print(f"    true 0  {cm[0,0]:>8} {cm[0,1]:>8}")
    print(f"    true 1  {cm[1,0]:>8} {cm[1,1]:>8}")
    fp, fn = int(cm[0, 1]), int(cm[1, 0])
    print(f"  false positives {fp}   false negatives {fn}")
    if fp + fn:
        print(f"  {100*fp/(fp+fn):.0f}% of errors are false positives")

    conf = np.maximum(prob, 1 - prob)
    wrong = pred != y
    if wrong.any() and (~wrong).any():
        print(f"\n  mean confidence: correct {conf[~wrong].mean():.3f}   "
              f"wrong {conf[wrong].mean():.3f}")
        if conf[wrong].mean() >= conf[~wrong].mean() - 0.01:
            print("  the model is as confident when wrong as when right —")
            print("  its probabilities carry little information")

    # calibration
    edges = np.linspace(0, 1, 11)
    ece = 0.0
    rows = []
    for i in range(10):
        m = (prob >= edges[i]) & (prob < edges[i + 1] + (i == 9) * 1e-9)
        if m.sum() == 0:
            continue
        c, a = prob[m].mean(), (y[m] == 1).mean()
        ece += m.sum() / len(y) * abs(c - a)
        rows.append((edges[i], edges[i + 1], int(m.sum()), c, a))
    print(f"\n  calibration (expected calibration error {ece:.4f})")
    for lo, hi, n_, c, a in rows:
        print(f"    [{lo:.1f},{hi:.1f})  n={n_:>5}  predicted {c:.3f}  "
              f"observed {a:.3f}")

    # length buckets — the long-document question
    if dev_rows is not None and ids is not None:
        idmap = {str(r[0]): r[1] for r in dev_rows}
        wl = np.array([len(idmap.get(str(i), "").split()) for i in ids])
        if (wl > 0).any():
            print("\n  accuracy by document length "
                  "(does the 512-token window break down?)")
            qs = sorted(set(np.quantile(wl[wl > 0], [0, .25, .5, .75, 1.0]).tolist()))
            if len(qs) < 2:
                print(f"    all dev docs are {int(qs[0])} words")
            else:
                accs = []
                for i in range(len(qs) - 1):
                    lo, hi = qs[i], qs[i + 1]
                    last = i == len(qs) - 2
                    m = (wl >= lo) & ((wl <= hi) if last else (wl < hi)) & (wl > 0)
                    if m.sum() == 0:
                        continue
                    a = (pred[m] == y[m]).mean()
                    accs.append(a)
                    print(f"    {int(lo):>6}-{int(hi):>6} words  n={m.sum():>5}  "
                          f"acc {a:.4f}")
                if len(accs) >= 2 and accs[0] - accs[-1] > 0.03:
                    print("    accuracy falls on the longest documents — the")
                    print("    window is missing decisive material in those cases")
    return {"ece": float(ece), "confusion": cm.tolist()}


# ════════════════════════════════════════════════════ one call

def experiment_report(cfg, name, info, metrics, dev_rows=None):
    """Everything about THIS experiment, and nothing about the others."""
    y = metrics["_y"]
    logits = metrics["_logits"]
    d = _load_pred(cfg, name)
    ids = d["ids"] if d is not None else None

    print("=" * 58)
    print(f"EXPERIMENT REPORT — {name}")
    print("=" * 58)
    print(f"  dev macro-F1 {metrics['macro_f1']:.4f}   "
          f"precision {metrics['precision']:.4f}   "
          f"recall {metrics['recall']:.4f}   "
          f"accuracy {metrics['accuracy']:.4f}")
    print(f"  per-class F1 {[round(x,4) for x in metrics['f1_per_class']]}")
    print(f"  trained {info.get('steps','?')} steps in "
          f"{info.get('train_seconds',0)/60:.1f} min")
    print()
    training_curve(info, metrics)
    vs_floors(metrics, y)
    balance_report(metrics, y)
    thr = threshold_sweep(logits, y)
    error_profile(logits, y, ids, dev_rows)
    print("\n" + "=" * 58)
    print("What this experiment demonstrated, and whether the technique was")
    print("worth it, is yours to write.")
    print("=" * 58)
    return thr


# ════════════════════════════════════════════════════ opt-in: all six

def compare_all(cfg):
    """Only for the final write-up, once several experiments are recorded."""
    path = Path(cfg.results_json)
    rows = json.loads(path.read_text()) if path.exists() else []
    if len(rows) < 2:
        print(f"{len(rows)} experiment(s) recorded — nothing to compare yet.")
        return rows
    rows = sorted(rows, key=lambda r: -r["macro_f1"])
    w = max(len(r["name"]) for r in rows) + 2
    print("=" * (w + 54))
    print("ALL EXPERIMENTS RECORDED SO FAR (dev)")
    print("=" * (w + 54))
    print(f"{'experiment':<{w}}{'macro-F1':>10}{'prec':>7}{'rec':>7}"
          f"{'train %':>9}{'mins':>7}")
    print("-" * (w + 54))
    for r in rows:
        print(f"{r['name']:<{w}}{r['macro_f1']:>10.4f}{r['precision']:>7.4f}"
              f"{r['recall']:>7.4f}{r['trainable_pct']:>9.3f}"
              f"{r['train_seconds']/60:>7.1f}")
    print("-" * (w + 54))
    base = next((r for r in rows if r["name"] == "01_full_finetune"), None)
    if base:
        print(f"\nretained vs full fine-tuning ({base['macro_f1']:.4f}):")
        for r in rows:
            if r["name"] == base["name"]:
                continue
            print(f"  {r['name']:<24}{100*r['macro_f1']/max(base['macro_f1'],1e-9):>7.1f}%"
                  f"   at {r['trainable_params']/max(base['trainable_params'],1):.4f}x "
                  f"the trainable parameters")
    return rows


def mcnemar(cfg, name_a, name_b):
    """Paired test on the same dev cases — use when writing the comparison."""
    pa, pb = _load_pred(cfg, name_a), _load_pred(cfg, name_b)
    if pa is None or pb is None:
        print("missing saved dev predictions for one of these runs")
        return
    assert np.array_equal(pa["y"], pb["y"])
    y = pa["y"]
    a_ok = pa["logits"].argmax(1) == y
    b_ok = pb["logits"].argmax(1) == y
    n01 = int((~a_ok & b_ok).sum()); n10 = int((a_ok & ~b_ok).sum())
    if n01 + n10 == 0:
        print("identical predictions")
        return
    chi2 = (abs(n10 - n01) - 1) ** 2 / (n01 + n10)
    p = math.erfc(math.sqrt(chi2 / 2))
    print(f"McNemar {name_a} vs {name_b}: only-A {n10}, only-B {n01}, "
          f"chi2 {chi2:.3f}, p {p:.4f} "
          f"({'significant' if p < 0.05 else 'not significant'})")
    return p